# LDA — supervised directions that separate classes

> Tutorial pair for [`lda.py`](lda.py).

## 1. Intuition
PCA asks "where does the data vary most?" — it never looks at the labels. **Fisher's
Linear Discriminant Analysis** asks a sharper question: *which directions push the
classes apart while keeping each class tight?* Project onto those directions and the
classes line up neatly, ready for a simple linear classifier.

## 2. Concept (the slide)
- Summarize each class by its **mean**; summarize spread with two scatter matrices:
  - **within-class** $S_W$ — how fuzzy each class is around its own mean,
  - **between-class** $S_B$ — how far the class means sit from the global mean.
- A good projection $\mathbf w$ makes $S_B$ large and $S_W$ small. Maximize their
  ratio (the **Fisher / Rayleigh quotient**).
- The optimum solves a **generalized eigenvalue problem**. At most $C-1$ useful
  axes exist for $C$ classes (because $S_B$ has rank $\le C-1$).
- With a shared-covariance Gaussian assumption, LDA is also a **linear classifier**.

## 3. Math derivation

**Scatter matrices.** With class means $\mathbf m_c$, global mean $\mathbf m$, counts $n_c$:

$$S_W=\sum_{c}\sum_{i\in c}(\mathbf x_i-\mathbf m_c)(\mathbf x_i-\mathbf m_c)^\top,
\qquad
S_B=\sum_{c} n_c(\mathbf m_c-\mathbf m)(\mathbf m_c-\mathbf m)^\top.$$

**Fisher criterion.** For a projection direction $\mathbf w$, the between- and
within-class variances of the projected data are $\mathbf w^\top S_B\mathbf w$ and
$\mathbf w^\top S_W\mathbf w$. Maximize their ratio:

$$J(\mathbf w)=\frac{\mathbf w^\top S_B\mathbf w}{\mathbf w^\top S_W\mathbf w}.$$

$J$ is scale-invariant, so fix $\mathbf w^\top S_W\mathbf w=1$ and use a Lagrangian
$\mathcal L=\mathbf w^\top S_B\mathbf w-\lambda(\mathbf w^\top S_W\mathbf w-1)$. Setting
$\nabla_{\mathbf w}\mathcal L=0$:

$$2S_B\mathbf w-2\lambda S_W\mathbf w=0\;\Longrightarrow\;\boxed{S_B\mathbf w=\lambda S_W\mathbf w}.$$

This **generalized eigenproblem** is equivalent to $S_W^{-1}S_B\mathbf w=\lambda\mathbf w$,
and at the optimum $J(\mathbf w)=\lambda$ — the eigenvalue *is* the separability. Pick
the top-$k$ eigenvectors.

**Whitening trick (what the code does).** $S_B\mathbf w=\lambda S_W\mathbf w$ is awkward
because $S_W^{-1}S_B$ is generally non-symmetric. Factor $S_W=S_W^{1/2}S_W^{1/2}$ and
substitute $\mathbf u=S_W^{1/2}\mathbf w$:

$$\big(S_W^{-1/2}S_B\,S_W^{-1/2}\big)\,\mathbf u=\lambda\,\mathbf u,$$

a **symmetric** standard eigenproblem (use `eigh`, numerically stable); recover
$\mathbf w=S_W^{-1/2}\mathbf u$.

**Two-class closed form.** With $C=2$, $S_B$ has rank 1 and the single optimal
direction is

$$\boxed{\;\mathbf w\propto S_W^{-1}(\mathbf m_1-\mathbf m_0)\;}$$

**LDA as a Gaussian classifier.** Assume each class is Gaussian with a *shared*
covariance $\Sigma=S_W/(n-C)$ and prior $\pi_c$. The log-posterior's quadratic term
cancels, leaving a **linear discriminant**

$$\delta_c(\mathbf x)=\mathbf x^\top\Sigma^{-1}\mathbf m_c-\tfrac12\mathbf m_c^\top\Sigma^{-1}\mathbf m_c+\log\pi_c,$$

and we predict $\arg\max_c\delta_c(\mathbf x)$.

**Regularization.** When $S_W$ is singular ($n<d$ or collinear features),
**shrink** it toward a sphere: $S_W\leftarrow(1-\gamma)S_W+\gamma\frac{\operatorname{tr}S_W}{d}I$.

## 4. NumPy implementation (generalized eigenproblem + two-class + Gaussian classifier)

In [ ]:
# ===== actual implementation from lda.py =====
from __future__ import annotations

import numpy as np

SEED = 0

import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def demo():
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    from sklearn.datasets import load_iris
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

    X, y = load_iris(return_X_y=True)

    lda = LDANumPy(n_components=2).fit(X, y)
    Z = lda.transform(X)
    print("LDA eigenvalues (separation per axis):", np.round(lda.eigenvalues_, 3))
    print("explained discriminability ratio:", np.round(lda.explained_variance_ratio_, 3))

    # Class separation in the 1D Fisher projection vs raw: ratio of between/within var.
    acc = (lda.predict(X) == y).mean()
    print(f"LDA-as-classifier training accuracy: {acc:.3f}")

    # Compare with sklearn (sign/scale may differ; compare absolute correlation of axis 1).
    sk = LinearDiscriminantAnalysis(n_components=2).fit(X, y)
    Zsk = sk.transform(X)
    corr = abs(np.corrcoef(Z[:, 0], Zsk[:, 0])[0, 1])
    print(f"|corr| of 1st LDA axis vs sklearn: {corr:.3f}")
    print(f"sklearn LDA training accuracy:      {sk.score(X, y):.3f}")

    # Two-class closed form (setosa vs rest collapsed to two classes).
    yb = (y > 0).astype(int)
    w = fisher_two_class(X, yb)
    proj = X @ w
    sep = abs(proj[yb == 0].mean() - proj[yb == 1].mean())
    within = proj[yb == 0].std() + proj[yb == 1].std()
    print(f"two-class Fisher separation/within-spread: {sep / (within + 1e-9):.3f}")

    # Torch path agrees with NumPy on the projected coordinates (up to sign).
    Zt, _ = lda_torch(X, y, n_components=2)
    agree = abs(np.corrcoef(Z[:, 0], Zt[:, 0])[0, 1])
    print(f"|corr| numpy vs torch 1st axis: {agree:.3f}")


class LDANumPy:
    r"""
    Fisher LDA. With class means :math:`m_c`, global mean :math:`m`, and
    per-class counts :math:`n_c`:

        within-class scatter   S_W = sum_c sum_{i in c} (x_i - m_c)(x_i - m_c)^T
        between-class scatter   S_B = sum_c n_c (m_c - m)(m_c - m)^T

    We seek directions :math:`w` maximizing the Rayleigh quotient

        J(w) = (w^T S_B w) / (w^T S_W w).

    Stationarity gives the generalized eigenproblem  S_B w = λ S_W w; the top
    eigenvectors (largest λ) are the discriminant axes. At most C-1 of them are
    nonzero because S_B has rank ≤ C-1.
    """

    def __init__(self, n_components=None, shrinkage=0.0):
        # shrinkage adds gamma*I to S_W (regularization toward a sphere)
        self.n_components = n_components
        self.shrinkage = shrinkage

    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        n_features = X.shape[1]
        mean_global = X.mean(0)

        S_W = np.zeros((n_features, n_features))   # within-class scatter
        S_B = np.zeros((n_features, n_features))   # between-class scatter
        self.means_ = {}
        for c in self.classes_:
            Xc = X[y == c]
            m_c = Xc.mean(0)
            self.means_[c] = m_c
            Xc_centered = Xc - m_c
            S_W += Xc_centered.T @ Xc_centered          # pooled scatter within c
            diff = (m_c - mean_global).reshape(-1, 1)
            S_B += len(Xc) * (diff @ diff.T)            # n_c * outer(mean diff)

        # Shrinkage / ridge: keep S_W invertible (vital when n < d or collinear).
        if self.shrinkage > 0:
            S_W = S_W + self.shrinkage * np.trace(S_W) / n_features * np.eye(n_features)

        # Solve S_B w = λ S_W w. Whiten by S_W: let A = S_W^{-1/2} S_B S_W^{-1/2},
        # a SYMMETRIC matrix, so eigh is stable; map eigenvectors back by S_W^{-1/2}.
        # eigh on S_W gives S_W = U diag(s) U^T, so S_W^{-1/2} = U diag(s^{-1/2}) U^T.
        s, U = np.linalg.eigh(S_W)
        s = np.maximum(s, 1e-12)                        # guard tiny/neg eigenvalues
        S_W_inv_half = U @ np.diag(1.0 / np.sqrt(s)) @ U.T
        A = S_W_inv_half @ S_B @ S_W_inv_half
        A = (A + A.T) / 2                               # symmetrize against fp drift
        eigvals, eigvecs = np.linalg.eigh(A)            # ascending order

        order = np.argsort(eigvals)[::-1]               # largest λ first
        eigvals = eigvals[order]
        eigvecs = eigvecs[:, order]
        W = S_W_inv_half @ eigvecs                      # back to original space
        # normalize columns to unit length for interpretability
        W = W / (np.linalg.norm(W, axis=0, keepdims=True) + 1e-12)

        max_comp = len(self.classes_) - 1               # rank(S_B) <= C-1
        k = self.n_components or max_comp
        k = min(k, max_comp, n_features)
        self.scalings_ = W[:, :k]
        self.eigenvalues_ = eigvals[:k]
        # explained "discriminability" ratio (separation captured per axis)
        pos = np.maximum(eigvals, 0)
        self.explained_variance_ratio_ = (pos[:k] / (pos.sum() + 1e-12))
        return self

    def transform(self, X):
        return np.asarray(X, float) @ self.scalings_

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

    def predict(self, X):
        """LDA as a classifier: nearest class mean in the *projected* space
        (equivalent to a shared-covariance Gaussian / linear discriminant)."""
        Z = self.transform(X)
        proj_means = np.stack([self.means_[c] @ self.scalings_ for c in self.classes_])
        d2 = ((Z[:, None, :] - proj_means[None, :, :]) ** 2).sum(2)
        return self.classes_[d2.argmin(1)]


def fisher_two_class(X, y):
    r"""Closed-form two-class Fisher direction  w ∝ S_W^{-1}(m_1 - m_0).

    For two classes the generalized eigenproblem collapses to this single
    direction (S_B has rank 1), which is why it has a tidy closed form."""
    X = np.asarray(X, float)
    y = np.asarray(y)
    classes = np.unique(y)
    assert len(classes) == 2, "two-class only"
    m0, m1 = X[y == classes[0]].mean(0), X[y == classes[1]].mean(0)
    S_W = np.zeros((X.shape[1],) * 2)
    for c, m in zip(classes, (m0, m1)):
        Xc = X[y == c] - m
        S_W += Xc.T @ Xc
    w = np.linalg.solve(S_W, m1 - m0)                   # S_W^{-1} (m1 - m0)
    return w / np.linalg.norm(w)

## 5. PyTorch implementation (whitening + symmetric eigensolver)

In [ ]:
# ===== actual implementation from lda.py =====
def lda_torch(X, y, n_components=None, shrinkage=0.0):
    """LDA via the generalized eigenproblem using torch.linalg.

    Same whitening trick as the NumPy version: form S_W^{-1/2} S_B S_W^{-1/2}
    and use the symmetric eigensolver (eigh)."""
    dev = get_device()
    Xt = torch.as_tensor(np.asarray(X, float), dtype=torch.float64, device=dev)
    y = np.asarray(y)
    classes = np.unique(y)
    d = Xt.shape[1]
    mean_global = Xt.mean(0)
    S_W = torch.zeros((d, d), dtype=torch.float64, device=dev)
    S_B = torch.zeros((d, d), dtype=torch.float64, device=dev)
    for c in classes:
        Xc = Xt[torch.as_tensor(y == c, device=dev)]
        m_c = Xc.mean(0)
        Xc_c = Xc - m_c
        S_W += Xc_c.T @ Xc_c
        diff = (m_c - mean_global).reshape(-1, 1)
        S_B += Xc.shape[0] * (diff @ diff.T)
    if shrinkage > 0:
        S_W = S_W + shrinkage * torch.trace(S_W) / d * torch.eye(d, dtype=torch.float64, device=dev)

    s, U = torch.linalg.eigh(S_W)
    s = torch.clamp(s, min=1e-12)
    S_W_inv_half = U @ torch.diag(1.0 / torch.sqrt(s)) @ U.T
    A = S_W_inv_half @ S_B @ S_W_inv_half
    A = (A + A.T) / 2
    eigvals, eigvecs = torch.linalg.eigh(A)
    order = torch.argsort(eigvals, descending=True)
    W = S_W_inv_half @ eigvecs[:, order]
    W = W / (W.norm(dim=0, keepdim=True) + 1e-12)
    k = (n_components or (len(classes) - 1))
    k = min(k, len(classes) - 1, d)
    W = W[:, :k]
    Z = Xt @ W
    return Z.cpu().numpy(), W.cpu().numpy()

## 6. Train / run — eigenvalues, accuracy, and comparison to sklearn & PCA

In [ ]:
demo()

## 7. Visualization — Iris projected by LDA vs PCA

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import lda as M

from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True); X = (X - X.mean(0)) / X.std(0)

Z = M.LDANumPy(n_components=2).fit_transform(X, y)          # supervised
Xc = X - X.mean(0); _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
Zp = Xc @ Vt[:2].T                                          # unsupervised PCA

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for c in np.unique(y):
    ax[0].scatter(Z[y == c, 0], Z[y == c, 1], s=18, label=f"class {c}")
    ax[1].scatter(Zp[y == c, 0], Zp[y == c, 1], s=18)
ax[0].set_title("LDA (supervised)"); ax[0].set_xlabel("LD1"); ax[0].set_ylabel("LD2")
ax[0].legend()
ax[1].set_title("PCA (unsupervised)"); ax[1].set_xlabel("PC1"); ax[1].set_ylabel("PC2")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- LDA is **supervised** — it uses labels to find class-separating axes; PCA does not.
- You get at most **$C-1$** discriminant directions (3 classes → 2 axes).
- $S_W$ must be invertible: **standardize**, and use **shrinkage** when $n<d$.
- The Gaussian-classifier view assumes **equal class covariances**; when they differ,
  use **QDA** (per-class covariance, quadratic boundary).
- LDA is **linear**; for curved class boundaries, combine with kernels or use
  nonlinear embeddings (t-SNE/UMAP) for *visualization* (not classification).